# Introduction to LCEL, Runnables, and LangServe

This notebook demonstrates the core concepts of the **LangChain Expression Language (LCEL)**, **Runnables**, **Chains**, and deployment with **LangServe**.
To make these concepts concrete, we will work through a practical case study: **An AI-Powered Customer Support Ticket Assistant**.

### Case Study Overview
Our support assistant will take raw support ticket text as input and execute the following automated steps:
1. **Parallel Triage**: Extract the ticket's **sentiment** (Angry, Neutral, Happy) and classify its **category** (Billing, Technical Support, General) in parallel.
2. **Conditional Routing**: Depending on the category, route the ticket to the appropriate prompt template designed for Billing specialists, Tech engineers, or General reps.
3. **Response Generation**: Generate a customized, context-aware auto-response using OpenAI's GPT models.
4. **LangServe API Deployment**: Expose this entire pipeline as a production-ready REST API with auto-generated documentation and client SDK endpoints.

---


## Prerequisites & Setup

Ensure you have the required packages installed and your OpenAI API Key configured in your environment.
Run the following cell to install the necessary packages.


In [ ]:
# Install libraries (if they are not already installed)
# %pip install langchain-openai langserve[all] fastapi uvicorn sse-starlette python-dotenv


In [ ]:
import os
from dotenv import load_dotenv

# Load environment variables
load_dotenv()

# Verify that OpenAI API key is set
if "OPENAI_API_KEY" in os.environ:
    print("[SUCCESS] OpenAI API Key is loaded successfully!")
else:
    print("[WARNING] OPENAI_API_KEY environment variable is missing. Please set it to run the model.")


## Part 1: LangChain Expression Language (LCEL) & Runnables

### What is LCEL?
**LangChain Expression Language (LCEL)** is a declarative way to compose LangChain components into chains. It was designed from day one to support:
- **Streaming support**: Get tokens as they are generated by the model.
- **Async support**: Run chains asynchronously for web servers.
- **Parallel execution**: Run multiple steps in parallel to reduce latency.
- **Automatic tracing**: Trace inputs and outputs using LangSmith.

### The Runnable Protocol
Almost every component in LangChain implements the **Runnable** interface. This standard interface includes:
- `invoke(input)`: Call the runnable on a single input.
- `stream(input)`: Stream back chunks of the response.
- `batch(inputs)`: Call the runnable on a list of inputs.
- `ainvoke`, `astream`, `abatch`: Async versions of the above.

Let's build a simple chain using the pipe operator (`|`) to see this in action.


In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_openai import ChatOpenAI

# 1. Initialize prompt template
simple_prompt = ChatPromptTemplate.from_template("Tell me a short, one-sentence joke about {topic}.")

# 2. Initialize the ChatOpenAI model
model = ChatOpenAI(model="gpt-4o-mini", temperature=0.7)

# 3. Initialize string output parser
output_parser = StrOutputParser()

# 4. Compose the chain using the pipe operator '|'
# Under the hood, this creates a RunnableSequence:
# input -> prompt -> model -> output_parser -> output string
joke_chain = simple_prompt | model | output_parser

print("--- 1. Runnable.invoke() ---")
# Call on a single input
response = joke_chain.invoke({"topic": "artificial intelligence"})
print(response)

print("\n--- 2. Runnable.batch() ---")
# Process multiple inputs in parallel
batch_responses = joke_chain.batch([{"topic": "pandas"}, {"topic": "coffee"}])
for resp in batch_responses:
    print("-", resp)

print("\n--- 3. Runnable.stream() ---")
# Stream tokens chunk-by-chunk
for chunk in joke_chain.stream({"topic": "programming"}):
    print(chunk, end="|", flush=True)
print()


## Part 2: Advanced LCEL: Chaining & Parallel Execution

In our Customer Support case study, we want to analyze two different aspects of a support ticket at the same time:
1. **Sentiment**: Is the customer angry, neutral, or happy?
2. **Category**: Is this ticket about Billing, Tech Support, or General inquiries?

Since these tasks do not depend on each other, running them sequentially wastes time. We can use `RunnableParallel` to execute them **in parallel**.

Let's define the prompts and the subchains for sentiment and category classification, and combine them.


In [ ]:
from langchain_core.runnables import RunnableParallel, RunnablePassthrough

# Define classification prompts
sentiment_prompt = ChatPromptTemplate.from_template(
    "Analyze the sentiment of this support ticket.\n"
    "Respond with exactly one word: 'Angry', 'Neutral', or 'Happy'.\n\n"
    "Ticket: {ticket_text}"
)

category_prompt = ChatPromptTemplate.from_template(
    "Classify this support ticket into one category: 'Billing', 'Technical Support', or 'General'.\n"
    "Respond with exactly the category name.\n\n"
    "Ticket: {ticket_text}"
)

# Compose independent subchains
sentiment_chain = sentiment_prompt | model | output_parser
category_chain = category_prompt | model | output_parser

# Combine them using RunnableParallel
# We also use RunnablePassthrough to keep a copy of the original ticket_text for the next step!
triage_chain = RunnableParallel(
    sentiment=sentiment_chain,
    category=category_chain,
    ticket_text=RunnablePassthrough() # Passes raw input forward
)

# Test our parallel triage chain
sample_ticket = "I was billed $50 twice on my credit card this morning! I demand an immediate refund!"
triage_result = triage_chain.invoke(sample_ticket)

print("Triage Result:")
print(triage_result)


## Part 3: Dynamic Chain Routing

Once we have analyzed the ticket's category and sentiment, we need to generate a response.
Different categories require different response styles:
- **Billing**: Needs empathy, reassurances, and clear billing terms.
- **Technical Support**: Needs a technical acknowledgment and reassurance that engineers are investigating.
- **General**: Needs a friendly, standard response.

In LCEL, we can route inputs to different templates dynamically using `RunnableLambda` (a custom routing function) or `RunnableBranch`.
A custom Python function is the recommended, most readable way.

Let's define the response templates and the routing function.


In [ ]:
from langchain_core.runnables import RunnableLambda

# Define response prompt templates
billing_template = ChatPromptTemplate.from_template(
    "You are an empathetic Billing Support Specialist at Acme Corp.\n"
    "The customer submitted this ticket with sentiment '{sentiment}':\n"
    "'{ticket_text}'\n\n"
    "Write a professional, reassuring email response addressing their concern. Focus on resolving the issue quickly."
)

tech_template = ChatPromptTemplate.from_template(
    "You are a Senior Tech Support Engineer at Acme Corp.\n"
    "The customer submitted this ticket with sentiment '{sentiment}':\n"
    "'{ticket_text}'\n\n"
    "Write a helpful response acknowledging the system issue and explaining that our developers are troubleshooting it."
)

general_template = ChatPromptTemplate.from_template(
    "You are a helpful Customer Service Representative at Acme Corp.\n"
    "The customer submitted this ticket with sentiment '{sentiment}':\n"
    "'{ticket_text}'\n\n"
    "Write a polite, generic response assisting them with their inquiry."
)

# Custom routing function
def route_by_category(inputs):
    category = inputs["category"].strip().lower()
    print(f"[Router] Detected Category: '{category}'")
    
    if "billing" in category:
        return billing_template
    elif "technical" in category or "tech" in category:
        return tech_template
    else:
        return general_template

# Combine the entire pipeline:
# 1. Parallel Triage (Extract sentiment/category and keep ticket_text)
# 2. Dynamic Router (Choose response prompt based on category)
# 3. OpenAI LLM (Generate response)
# 4. StrOutputParser (Parse output text)
full_support_chain = triage_chain | RunnableLambda(route_by_category) | model | output_parser

# Run the complete chain
print("--- Test Case 1: Billing Issue ---")
ticket_1 = "I noticed a double payment invoice on my subscription billing list."
response_1 = full_support_chain.invoke(ticket_1)
print("\nGenerated Auto-Response:\n", response_1)

print("\n" + "="*80 + "\n")

print("--- Test Case 2: Tech Issue ---")
ticket_2 = "Every time I try to login, the screen hangs on a white page and throws a 500 error."
response_2 = full_support_chain.invoke(ticket_2)
print("\nGenerated Auto-Response:\n", response_2)


## Part 4: Deployment with LangServe

### What is LangServe?
**LangServe** is a library that helps developers deploy LangChain runnables and chains as a **REST API** using **FastAPI**.
It automatically adds:
- Schema validation for inputs and outputs (using Pydantic).
- Streaming endpoints (`/stream`, `/stream_log`).
- Batch execution endpoints (`/batch`).
- An interactive web client playground (`/playground`) out-of-the-box!

### Deployment Architecture
Below is the typical layout of a LangServe application:
1. Define your LangChain runnable/chain.
2. Initialize a FastAPI app.
3. Call `add_routes(app, chain, path="/endpoint")`.
4. Start the server using Uvicorn.

We have created a standalone Python script in this directory called [support_server.py](file:///d:/Training/Edureka/Module%203/3.%20Demo%20%20M_III/support_server.py) which contains the complete server setup.


In [ ]:
# Let's inspect our server script:
if os.path.exists("support_server.py"):
    with open("support_server.py", "r", encoding="utf-8") as f:
        print(f.read())
else:
    print("[WARNING] support_server.py was not found. Please ensure it is present in the workspace.")


### How to Run and Interact with the API

#### 1. Running the Server
To run the server script, execute this command in your terminal:
```bash
python support_server.py
```
This will start the FastAPI application on port `8000`.

#### 2. Accessing the Playground
Once the server is running, open your web browser and navigate to:
`http://localhost:8000/support/playground/`

You will see a beautiful visual interface where you can input ticket text, click **Submit**, and see the response generated live!

#### 3. Calling the API Programmatically
You can invoke the LangServe API from a remote Python client using LangChain's `RemoteRunnable` class. It acts exactly like a local runnable, implementing the same `invoke`, `batch`, and `stream` methods!

Let's look at a client-side example.


In [ ]:
from langserve import RemoteRunnable

# Initialize the remote runnable pointing to our LangServe endpoint
remote_chain = RemoteRunnable("http://localhost:8000/support")

# (Note: This cell will fail if your support_server.py is not running on localhost:8000!)
try:
    print("Sending ticket to remote LangServe API...")
    remote_response = remote_chain.invoke("Hi, I want to cancel my account subscription. Please process it.")
    print("\nAPI Response:")
    print(remote_response)
except Exception as e:
    print(f"\n[CLIENT INFO] The server is not running on localhost:8000.")
    print("Start the server by running 'python support_server.py' in a separate terminal window and try again.")
    print(f"Exception message: {e}")


## Summary & Conclusion

In this notebook, you learned:
1. **LCEL (LangChain Expression Language)**: A declarative way to chain components using the `|` operator.
2. **Runnable Protocol**: The standard interface (`invoke`, `batch`, `stream`) implemented by LangChain components.
3. **Advanced Composability**:
   - `RunnableParallel` for running tasks concurrently.
   - `RunnableLambda` for dynamic routing based on values.
4. **LangServe**: Exposing LangChain runnables as API endpoints with FastAPI.
5. **Remote Consumability**: Interacting with LangServe APIs via `RemoteRunnable` as if they were local chains.

You are now ready to build complex, production-ready, auto-routing agents and deploy them as microservices!

---
*Created for Edureka Module 3 - LangChain LCEL & LangServe Demonstration*
